In [1]:
#pip install -q -U \ langchain \ langchain-core \ langchain-community \ langchain-google-genai \ faiss-cpu

In [2]:
2+2

4

In [3]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from dotenv import load_dotenv

C:\Users\vaish\AppData\Local\Temp\ipykernel_5260\3149061991.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
load_dotenv()

api_key=os.getenv('GEMINI_API_KEY')


In [5]:
#2.Initialize the embedding model

embedder=GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)



In [6]:
#3 Initialize chat model

llm=ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.0

)

In [ ]:
#4.Mount the faiss

DB_DIR="faiss_extract_once_db"

vectorstore=FAISS.load_local(
    folder_path=DB_DIR,
    embeddings=embedder,
    allow_dangerous_deserialization=True
)

In [8]:
#5.Convert the database inot langchain retriever

retriever=vectorstore.as_retriever(search_kwags={"k":3})

In [ ]:
#6 The context formatter

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    

In [11]:
#7. Define the RAG Mega-Prompt

template = """

You are an elite MLOps auditing algorithm for the 'Extract Once, Judge Many' project.
Your core directive is to answer the user's query using STRICTLY the context provided below.

CRITICAL RULES:

1. Anti-Hallucination Protocol: If the answer is not explicitly contained in the context, you must output: "I cannot answer this based on the provided corporate documents." Do not attempt to guess.
2. Do not use your pre-trained internet knowledge.
3. Keep your answer professional, highly concise, and cite the author if available.

====================

RETRIEVED CONTEXT:

{context}

====================

USER QUERY: {question}

FINAL ANSWER:

"""

rag_prompt=PromptTemplate.from_template(template)

In [20]:
#8.THE LCEL CHAIN--langchain expression languague

#retriever|prompt|llm|parser

rag_chain=(
    {"context":retriever | format_docs,"question":RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

#takes the query grabs the users query and passes 

In [21]:
#9. Execute the valid query

user_question="What kind of architecture are we talking about?"

final_response=rag_chain.invoke(user_question)
print(f"Ai response is {final_response}")

Ai response is FINAL ANSWER:

Tony has proposed moving to a distributed architecture.


C:\Users\vaish\AppData\Local\Temp\ipykernel_5260\77622800.py:5: RuntimeWarning: coroutine 'RunnableSequence.ainvoke' was never awaited
  final_response=rag_chain.invoke(user_question)


In [23]:
user_question="what is newton school?"

final_response=rag_chain.invoke(user_question)
print(f"Ai response is {final_response}")

Ai response is I cannot answer this based on the provided corporate documents.


**COMPLETE RAG PIPELINE**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader 
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

import os
load_dotenv()
api_key=os.getenv("GEMINI_API_KEY")

#loading
loader=PyPDFLoader("attension_is_all_you_need.pdf")

pages=loader.load()

#splitting

splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    separators=["\n\n","\n",", "," ",""]

)

chunks=splitter.split_documents(pages)

# for i ,chunk in enumerate(chunks):
#     print(f"chunk{i+1}")
#     print(chunk.page_content)
#     print("*"*100)
#     print(chunk.metadata)

#embedding

embedder=HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

# or 
# embedder=GoogleGenerativeAIEmbeddings(
#     model="models/gemini-embedding-001",
#     api_key=api_key
# )

# text_embeddings=embedder.embed_documents(
#     [doc.page_content for doc in chunks]
# )

# print(text_embeddings[5])
# print(len(text_embeddings[0]))


#vector db

vector_store=FAISS.from_documents(chunks,embedder)

base_retriever=vector_store.as_retriever(search_kwargs={"k":5})

#results=vector_store.similarity_search(query)

Db_dir="faiss_attension"
vector_store.save_local(Db_dir)



response_db=FAISS.load_local(
    folder_path=Db_dir,
    embeddings=embedder,
    allow_dangerous_deserialization=True
)

# print(response_db.similarity_search("what is encoder"))
# print("**"*40)
# print(vector_store.similarity_search("what is encoder"))


#cross encoder
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

cross_encoder_model=HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

reranker=CrossEncoderReranker(model=cross_encoder_model,top_n=3)

compression_retriever=ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever #or base_retriver=response_db.as_retriever(search_kwargs={"k": 5}) if u do del vector_Store in above deleteing form RAM
)

#prompt
system_prompt="""
you are a expert in finding answers based on provide context and question

context:
{context}

question:
{query}
"""

prompt=ChatPromptTemplate.from_template(system_prompt)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    

#llm

llm=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.0,
    api_key=api_key
)

#chain
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain=(
    {
        "context":compression_retriever|format_docs,
        "query":RunnablePassthrough()
    }
        |prompt
        |llm
        |StrOutputParser()
    
)

query="what is self attention mechanism"

response=chain.invoke(query)
print(response)






Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2336.94it/s]
c:\Users\vaish\agenticaisep\.venv13\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided context, the self-attention mechanism (sometimes called intra-attention) is **an attention mechanism relating different positions** and **connects all positions with a constant number of sequentially [operations]** (as noted in Table 1).


In [ ]:
compression_retriever.invoke(query) #retrived content

[Document(id='7510db35-9cba-49a0-846d-ed36d9ea6090', metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existi

after restart do below only

In [ ]:
response_db = FAISS.load_local(
    folder_path="faiss_attension",
    embeddings=embedder,
    allow_dangerous_deserialization=True
)

base_retriever = response_db.as_retriever(
    search_kwargs={"k": 5}
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

system_prompt="""
you are a expert in finding answers based on provide context and question

context:
{context}

question:
{query}
"""

prompt=ChatPromptTemplate.from_template(system_prompt)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    

#llm

llm=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.0,
    api_key=api_key
)

#chain
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain=(
    {
        "context":compression_retriever|format_docs,
        "query":RunnablePassthrough()
    }
        |prompt
        |llm
        |StrOutputParser()
    
)

query="what is self attention mechanism"

response=chain.invoke(query)
print(response)
